# LLM Infrastructure Cost Analysis
## Should We Self-Host or Stay on Bedrock?

---

### Context

Our AI appeals application generates insurance appeal documents using Claude Haiku 4.5 via AWS Bedrock.  
Based on production telemetry from this week's ACME workload, we observe the following **per-appeal token profile**:

| Metric | Tokens |
|--------|--------|
| Input tokens | ~58,500 |
| Output tokens | ~8,500 |
| Cache read tokens | ~157,000 |
| Cache write tokens | ~48,000 |
| **Total** | **~272,000** |

At current Bedrock pricing, this costs approximately **$0.18 per appeal**.

### The Problem

API pricing scales **linearly** — every additional appeal costs exactly the same. As we onboard customers and scale toward **300K–1M+ appeals/month**, Bedrock costs alone will reach **$50K–$180K/month** on Haiku, or **$160K–$530K/month** if we upgrade to Sonnet.

Self-hosted infrastructure has a **fixed** compute cost that stays roughly constant regardless of how many appeals we process (until we hit capacity and add another node). At sufficient volume, the cost per appeal drops asymptotically toward zero.

**The question: at what volume does self-hosting become cheaper?**

The visualizations below answer this interactively.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, FloatSlider, Layout, HBox, VBox

from llm_cost import TokenUsage, CLAUDE_HAIKU_45, CLAUDE_SONNET_45
from self_hosted_cost import (
    CONFIGS,
    PALETTE,
    SECONDS_PER_MONTH,
    G6E_48XLARGE,
    P4DE_24XLARGE,
    P5_48XLARGE,
    LLAMA_3_3_70B,
    estimate_throughput,
)

slider_layout = Layout(width="600px")
slider_style = {"description_width": "160px"}

---

## 0. Throughput Requirements — "What Do We Actually Need?"

Before comparing costs, we need to understand the **throughput demand** our workload places on infrastructure. This determines what hardware is actually viable.

The key question: *"Given X appeals/month with Y output tokens each, how many tokens/second must our cluster sustain?"*

For self-hosted LLMs, the bottleneck is **decode throughput** — the rate at which the model generates output tokens. This is constrained by GPU memory bandwidth (the model weights must be read from VRAM for every single generated token).

The estimator below derives throughput from first principles:

1. **Memory feasibility** — Can the model weights + KV cache fit in VRAM?
2. **Max batch size** — How many concurrent requests fit given KV cache memory at our context length?
3. **Decode throughput** — `batch_size × memory_bandwidth / model_weight_size` (memory-bandwidth bound)
4. **Required throughput** — `appeals/month × output_tokens/appeal / seconds/month`

> Adjust sliders to see how volume and context length affect what hardware you need.

In [2]:
@interact(
    output_tokens=IntSlider(
        value=8_500,
        min=1_000,
        max=50_000,
        step=500,
        description="Output Tokens/Appeal",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
    context_length=IntSlider(
        value=60_000,
        min=10_000,
        max=200_000,
        step=5_000,
        description="Context Length (tok)",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
    peak_multiplier=FloatSlider(
        value=2.0,
        min=1.0,
        max=5.0,
        step=0.5,
        description="Peak/Avg Ratio",
        style=slider_style,
        layout=slider_layout,
        readout_format=".1f",
    ),
)
def plot_throughput_requirements(output_tokens, context_length, peak_multiplier):
    appeals = np.arange(0, 2_000_001, 5_000)

    # Required sustained decode throughput (output tok/s)
    required_avg = (appeals * output_tokens) / SECONDS_PER_MONTH
    required_peak = required_avg * peak_multiplier

    # Estimate hardware capacity from specs
    hw_configs = [
        (
            "8\u00d7L40S (INT8)",
            G6E_48XLARGE,
            "int8",
            PALETTE["L40S \u2014 Llama 70B INT8"],
        ),
        (
            "8\u00d7A100 80GB (FP16)",
            P4DE_24XLARGE,
            "fp16",
            PALETTE["A100 \u2014 Llama 70B FP16"],
        ),
        (
            "8\u00d7H100 80GB (FP16)",
            P5_48XLARGE,
            "fp16",
            PALETTE["H100 \u2014 Llama 70B FP16"],
        ),
    ]

    fig, ax = plt.subplots(figsize=(13, 7))

    # Demand lines
    ax.plot(
        appeals,
        required_avg,
        color="#2c3e50",
        linewidth=2.5,
        label=f"Required sustained throughput ({output_tokens:,} tok/appeal)",
    )
    ax.plot(
        appeals,
        required_peak,
        color="#2c3e50",
        linewidth=1.5,
        linestyle=":",
        alpha=0.7,
        label=f"Peak demand ({peak_multiplier:.1f}\u00d7 average)",
    )
    ax.fill_between(appeals, required_avg, required_peak, alpha=0.06, color="#2c3e50")

    # Hardware capacity lines + info table
    print("\n  Throughput Estimation (Llama 3.3 70B, roofline model):")
    print(f"  Context length: {context_length:,} tokens")
    print(
        f"  {'Config':<25s} {'VRAM Used':<12s} {'Max Batch':<12s} {'Decode tok/s':<15s} {'Handles (avg)'}"
    )
    print(
        f"  {'\u2500' * 25} {'\u2500' * 12} {'\u2500' * 12} {'\u2500' * 15} {'\u2500' * 15}"
    )

    for label, instance, quant, color in hw_configs:
        est = estimate_throughput(instance, LLAMA_3_3_70B, quant, context_length)
        capacity = est["batched_decode_tok_s"]

        ax.axhline(
            capacity,
            color=color,
            linewidth=2.5,
            linestyle="--",
            alpha=0.85,
            label=f"{label} capacity: {capacity:,.0f} tok/s (batch={est['max_batch_size']})",
        )

        # Light shading for capacity band
        ax.fill_between(appeals, 0, capacity, alpha=0.04, color=color)

        # How many appeals/month can this handle?
        max_appeals_avg = capacity * SECONDS_PER_MONTH / output_tokens
        model_mem = est["model_memory_gb"]
        print(
            f"  {label:<25s} {model_mem:<12.0f} GB  {est['max_batch_size']:<12d} "
            f"{capacity:<15,.0f} {max_appeals_avg:,.0f}/mo"
        )

    # Volume annotations
    ax.axvline(300_000, color="gray", linestyle=":", alpha=0.6, linewidth=1)
    ax.axvline(1_000_000, color="gray", linestyle=":", alpha=0.4, linewidth=1)
    ax.text(
        300_000, ax.get_ylim()[1] * 0.02, " 300K", fontsize=8, color="gray", va="bottom"
    )
    ax.text(
        1_000_000, ax.get_ylim()[1] * 0.02, " 1M", fontsize=8, color="gray", va="bottom"
    )

    # Formatting
    ax.set_xlabel("Appeals per Month", fontsize=12)
    ax.set_ylabel("Output Tokens per Second (sustained)", fontsize=12)
    ax.set_title(
        "Throughput Demand vs. Hardware Capacity (Llama 3.3 70B)",
        fontsize=13,
        fontweight="bold",
    )
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, _: f"{x / 1e6:.1f}M" if x >= 1e6 else f"{x / 1e3:.0f}K"
        )
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.set_xlim(0, 2_000_000)
    ax.set_ylim(0, None)
    ax.legend(loc="upper left", fontsize=9, framealpha=0.95)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

    # Key insight
    req_at_300k = (300_000 * output_tokens) / SECONDS_PER_MONTH
    req_at_1m = (1_000_000 * output_tokens) / SECONDS_PER_MONTH
    print(f"\n  Required sustained decode throughput:")
    print(
        f"    At 300K appeals/mo: {req_at_300k:,.0f} tok/s (peak: {req_at_300k * peak_multiplier:,.0f})"
    )
    print(
        f"    At 1M appeals/mo:   {req_at_1m:,.0f} tok/s (peak: {req_at_1m * peak_multiplier:,.0f})"
    )

interactive(children=(IntSlider(value=8500, description='Output Tokens/Appeal', layout=Layout(width='600px'), …

---

## 1. Monthly Total Cost vs. Appeal Volume — "The Crossover"

This is the fundamental chart. The **straight diagonal lines** represent Bedrock API costs — they scale linearly forever. The **step-function lines** represent self-hosted GPU clusters — they stay flat until capacity is reached, then jump by one node's cost.

The point where a self-hosted line drops **below** an API line is the crossover — beyond that volume, self-hosting is cheaper.

> **Use the sliders** to adjust the per-appeal token profile and see how the crossover shifts.

In [3]:
@interact(
    input_tokens=IntSlider(
        value=58_500,
        min=5_000,
        max=200_000,
        step=1_000,
        description="Input Tokens/Appeal",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
    output_tokens=IntSlider(
        value=8_500,
        min=1_000,
        max=50_000,
        step=500,
        description="Output Tokens/Appeal",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
    cache_read_tokens=IntSlider(
        value=157_000,
        min=0,
        max=500_000,
        step=5_000,
        description="Cache Read Tokens",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
    cache_write_tokens=IntSlider(
        value=48_000,
        min=0,
        max=150_000,
        step=1_000,
        description="Cache Write Tokens",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
)
def plot_crossover(input_tokens, output_tokens, cache_read_tokens, cache_write_tokens):
    appeals = np.arange(0, 2_000_001, 5_000)

    # ── API costs (linear) ──
    usage = TokenUsage(
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        cache_read_tokens=cache_read_tokens,
        cache_write_tokens=cache_write_tokens,
    )

    haiku_cpa = CLAUDE_HAIKU_45.calculate_cost(usage).total_cost
    sonnet_cpa = CLAUDE_SONNET_45.calculate_cost(usage).total_cost

    haiku_monthly = appeals * haiku_cpa
    sonnet_monthly = appeals * sonnet_cpa

    # ── Self-hosted costs (step function) ──
    fig, ax = plt.subplots(figsize=(13, 6.5))

    # API lines
    ax.plot(
        appeals,
        haiku_monthly,
        color=PALETTE["claude-haiku-4.5"],
        linewidth=2.5,
        label=f"Bedrock Haiku 4.5 (${haiku_cpa:.3f}/appeal)",
    )
    ax.plot(
        appeals,
        sonnet_monthly,
        color=PALETTE["claude-sonnet-4.5"],
        linewidth=2.5,
        label=f"Bedrock Sonnet 4.5 (${sonnet_cpa:.3f}/appeal)",
    )

    # Self-hosted lines
    for key, config in CONFIGS.items():
        costs = config.monthly_cost_curve(appeals, output_tokens)
        ax.step(
            appeals,
            costs,
            where="post",
            color=PALETTE[key],
            linewidth=2,
            linestyle="--",
            label=f"{config.label} (${config.instance_monthly_cost:,.0f}/node)",
        )

    # ── Annotations ──
    ax.axvline(300_000, color="gray", linestyle=":", alpha=0.7, linewidth=1.2)
    ax.text(
        310_000,
        ax.get_ylim()[1] * 0.92,
        "Initial target\n300K/mo",
        fontsize=9,
        color="gray",
        va="top",
    )

    ax.axvline(1_000_000, color="gray", linestyle=":", alpha=0.5, linewidth=1.2)
    ax.text(
        1_010_000,
        ax.get_ylim()[1] * 0.92,
        "Growth target\n1M/mo",
        fontsize=9,
        color="gray",
        va="top",
    )

    # ── Formatting ──
    ax.set_xlabel("Appeals per Month", fontsize=12)
    ax.set_ylabel("Monthly Cost (USD)", fontsize=12)
    ax.set_title(
        "Monthly Infrastructure Cost: Bedrock API vs. Self-Hosted GPU Clusters",
        fontsize=13,
        fontweight="bold",
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, _: f"{x / 1e6:.1f}M" if x >= 1e6 else f"{x / 1e3:.0f}K"
        )
    )
    ax.set_xlim(0, 2_000_000)
    ax.set_ylim(0, None)
    ax.legend(loc="upper left", fontsize=9.5, framealpha=0.95)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

    # ── Summary stats ──
    print(f"\n  At 300K appeals/month:")
    print(f"    Bedrock Haiku:    ${300_000 * haiku_cpa:>12,.0f}/mo")
    print(f"    Bedrock Sonnet:   ${300_000 * sonnet_cpa:>12,.0f}/mo")
    for key, config in CONFIGS.items():
        c = config.monthly_cost(300_000, output_tokens)
        n = config.required_instances(300_000, output_tokens)
        print(f"    {config.label}: ${c:>8,.0f}/mo ({n} node{'s' if n > 1 else ''})")

    print(f"\n  At 1M appeals/month:")
    print(f"    Bedrock Haiku:    ${1_000_000 * haiku_cpa:>12,.0f}/mo")
    print(f"    Bedrock Sonnet:   ${1_000_000 * sonnet_cpa:>12,.0f}/mo")
    for key, config in CONFIGS.items():
        c = config.monthly_cost(1_000_000, output_tokens)
        n = config.required_instances(1_000_000, output_tokens)
        print(f"    {config.label}: ${c:>8,.0f}/mo ({n} node{'s' if n > 1 else ''})")

interactive(children=(IntSlider(value=58500, description='Input Tokens/Appeal', layout=Layout(width='600px'), …

---

## 2. Cost Per Appeal vs. Volume — "The Efficiency Curve"

API pricing gives you a **flat horizontal line** — no volume discount, no economies of scale. Every appeal costs the same whether you process 1 or 1 million.

Self-hosted infrastructure produces a **declining curve** — fixed compute cost is amortized across more appeals. The more you process, the cheaper each one becomes.

This chart makes the economic argument viscerally clear: at scale, API pricing becomes an increasingly expensive premium over what the compute actually costs.

In [4]:
@interact(
    input_tokens=IntSlider(
        value=58_500,
        min=5_000,
        max=200_000,
        step=1_000,
        description="Input Tokens/Appeal",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
    output_tokens=IntSlider(
        value=8_500,
        min=1_000,
        max=50_000,
        step=500,
        description="Output Tokens/Appeal",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
    cache_read_tokens=IntSlider(
        value=157_000,
        min=0,
        max=500_000,
        step=5_000,
        description="Cache Read Tokens",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
    cache_write_tokens=IntSlider(
        value=48_000,
        min=0,
        max=150_000,
        step=1_000,
        description="Cache Write Tokens",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
)
def plot_cost_per_appeal(
    input_tokens, output_tokens, cache_read_tokens, cache_write_tokens
):
    # Start from 10K to avoid near-infinity at low volumes
    appeals = np.arange(10_000, 2_000_001, 5_000)

    usage = TokenUsage(
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        cache_read_tokens=cache_read_tokens,
        cache_write_tokens=cache_write_tokens,
    )

    haiku_cpa = CLAUDE_HAIKU_45.calculate_cost(usage).total_cost
    sonnet_cpa = CLAUDE_SONNET_45.calculate_cost(usage).total_cost

    fig, ax = plt.subplots(figsize=(13, 6.5))

    # API flat lines
    ax.axhline(
        haiku_cpa,
        color=PALETTE["claude-haiku-4.5"],
        linewidth=2.5,
        label=f"Bedrock Haiku 4.5 \u2014 ${haiku_cpa:.4f}/appeal (constant)",
    )
    ax.axhline(
        sonnet_cpa,
        color=PALETTE["claude-sonnet-4.5"],
        linewidth=2.5,
        label=f"Bedrock Sonnet 4.5 \u2014 ${sonnet_cpa:.4f}/appeal (constant)",
    )

    # Self-hosted declining curves
    for key, config in CONFIGS.items():
        cpa_curve = config.cost_per_appeal_curve(appeals, output_tokens)
        ax.plot(
            appeals,
            cpa_curve,
            color=PALETTE[key],
            linewidth=2,
            linestyle="--",
            label=f"{config.label}",
        )

    # ── Annotations ──
    ax.axvline(300_000, color="gray", linestyle=":", alpha=0.7, linewidth=1.2)
    ax.text(310_000, sonnet_cpa * 0.7, "300K/mo", fontsize=9, color="gray")

    ax.axvline(1_000_000, color="gray", linestyle=":", alpha=0.5, linewidth=1.2)
    ax.text(1_010_000, sonnet_cpa * 0.7, "1M/mo", fontsize=9, color="gray")

    # ── Formatting ──
    ax.set_xlabel("Appeals per Month", fontsize=12)
    ax.set_ylabel("Cost per Appeal (USD)", fontsize=12)
    ax.set_title(
        "Cost per Appeal: Fixed API Pricing vs. Declining Self-Hosted Cost",
        fontsize=13,
        fontweight="bold",
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.3f}"))
    ax.xaxis.set_major_formatter(
        mticker.FuncFormatter(
            lambda x, _: f"{x / 1e6:.1f}M" if x >= 1e6 else f"{x / 1e3:.0f}K"
        )
    )
    ax.set_xlim(10_000, 2_000_000)
    ax.set_ylim(0, sonnet_cpa * 1.4)
    ax.legend(loc="upper right", fontsize=9.5, framealpha=0.95)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

    # Summary
    print(f"\n  Cost per appeal at 300K/month:")
    print(f"    Bedrock Haiku:   ${haiku_cpa:.4f}")
    print(f"    Bedrock Sonnet:  ${sonnet_cpa:.4f}")
    for key, config in CONFIGS.items():
        c = config.cost_per_appeal(300_000, output_tokens)
        savings_vs_haiku = (1 - c / haiku_cpa) * 100
        print(f"    {config.label}: ${c:.4f}  ({savings_vs_haiku:+.0f}% vs Haiku)")

    print(f"\n  Cost per appeal at 1M/month:")
    print(f"    Bedrock Haiku:   ${haiku_cpa:.4f}")
    print(f"    Bedrock Sonnet:  ${sonnet_cpa:.4f}")
    for key, config in CONFIGS.items():
        c = config.cost_per_appeal(1_000_000, output_tokens)
        savings_vs_haiku = (1 - c / haiku_cpa) * 100
        print(f"    {config.label}: ${c:.4f}  ({savings_vs_haiku:+.0f}% vs Haiku)")

interactive(children=(IntSlider(value=58500, description='Input Tokens/Appeal', layout=Layout(width='600px'), …

---

## 3. 12-Month Projected Savings — "The Money Chart"

Given an expected starting volume and monthly growth rate, this chart projects **cumulative spend** over the next 12 months under two scenarios:

1. **Stay on Bedrock API** — pay per token, costs grow with volume  
2. **Switch to self-hosted** — pay for compute, costs grow only when capacity is exceeded

The **shaded area** between the lines represents total money saved by switching. This is the number that matters for the business case.

> Adjust the growth rate slider to model conservative vs. aggressive scaling scenarios.

In [5]:
@interact(
    starting_volume=IntSlider(
        value=300_000,
        min=50_000,
        max=1_000_000,
        step=25_000,
        description="Starting Appeals/Mo",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
    monthly_growth_pct=FloatSlider(
        value=15.0,
        min=0.0,
        max=50.0,
        step=1.0,
        description="Monthly Growth %",
        style=slider_style,
        layout=slider_layout,
        readout_format=".0f",
    ),
    output_tokens=IntSlider(
        value=8_500,
        min=1_000,
        max=50_000,
        step=500,
        description="Output Tokens/Appeal",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
    cache_read_tokens=IntSlider(
        value=157_000,
        min=0,
        max=500_000,
        step=5_000,
        description="Cache Read Tokens",
        style=slider_style,
        layout=slider_layout,
        readout_format=",",
    ),
)
def plot_savings_trajectory(
    starting_volume,
    monthly_growth_pct,
    output_tokens,
    cache_read_tokens,
):
    months = np.arange(1, 13)
    growth_factor = 1 + monthly_growth_pct / 100
    volumes = starting_volume * (growth_factor ** (months - 1))

    # Fixed token assumptions for API cost calc
    input_tokens = 58_500
    cache_write_tokens = 48_000

    usage = TokenUsage(
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        cache_read_tokens=cache_read_tokens,
        cache_write_tokens=cache_write_tokens,
    )
    haiku_cpa = CLAUDE_HAIKU_45.calculate_cost(usage).total_cost
    sonnet_cpa = CLAUDE_SONNET_45.calculate_cost(usage).total_cost

    # Monthly costs
    haiku_monthly = volumes * haiku_cpa
    sonnet_monthly = volumes * sonnet_cpa

    # Pick the best-value self-hosted config (L40S)
    config = CONFIGS["L40S \u2014 Llama 70B INT8"]
    self_hosted_monthly = np.array(
        [config.monthly_cost(int(v), output_tokens) for v in volumes]
    )

    # Cumulative
    haiku_cumulative = np.cumsum(haiku_monthly)
    sonnet_cumulative = np.cumsum(sonnet_monthly)
    self_hosted_cumulative = np.cumsum(self_hosted_monthly)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # ── Left panel: Monthly spend ──
    ax1.plot(
        months,
        haiku_monthly / 1000,
        color=PALETTE["claude-haiku-4.5"],
        linewidth=2.5,
        marker="o",
        markersize=4,
        label="Bedrock Haiku 4.5",
    )
    ax1.plot(
        months,
        sonnet_monthly / 1000,
        color=PALETTE["claude-sonnet-4.5"],
        linewidth=2.5,
        marker="o",
        markersize=4,
        label="Bedrock Sonnet 4.5",
    )
    ax1.plot(
        months,
        self_hosted_monthly / 1000,
        color=PALETTE["L40S \u2014 Llama 70B INT8"],
        linewidth=2.5,
        marker="s",
        markersize=4,
        linestyle="--",
        label=f"Self-Hosted ({config.label})",
    )

    ax1.set_xlabel("Month", fontsize=11)
    ax1.set_ylabel("Monthly Cost ($K)", fontsize=11)
    ax1.set_title("Monthly Spend Over 12 Months", fontsize=12, fontweight="bold")
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}K"))
    ax1.set_xticks(months)
    ax1.legend(fontsize=9, loc="upper left")
    ax1.grid(True, alpha=0.25)

    # ── Right panel: Cumulative spend with savings shading ──
    ax2.plot(
        months,
        haiku_cumulative / 1e6,
        color=PALETTE["claude-haiku-4.5"],
        linewidth=2.5,
        marker="o",
        markersize=4,
        label="Bedrock Haiku 4.5",
    )
    ax2.plot(
        months,
        sonnet_cumulative / 1e6,
        color=PALETTE["claude-sonnet-4.5"],
        linewidth=2.5,
        marker="o",
        markersize=4,
        label="Bedrock Sonnet 4.5",
    )
    ax2.plot(
        months,
        self_hosted_cumulative / 1e6,
        color=PALETTE["L40S \u2014 Llama 70B INT8"],
        linewidth=2.5,
        marker="s",
        markersize=4,
        linestyle="--",
        label=f"Self-Hosted ({config.label})",
    )

    # Shade savings vs Haiku
    ax2.fill_between(
        months,
        self_hosted_cumulative / 1e6,
        haiku_cumulative / 1e6,
        alpha=0.15,
        color=PALETTE["claude-haiku-4.5"],
        label="_nolegend_",
    )

    total_savings_haiku = haiku_cumulative[-1] - self_hosted_cumulative[-1]
    total_savings_sonnet = sonnet_cumulative[-1] - self_hosted_cumulative[-1]

    # Annotate savings
    mid_month = 8
    mid_y = (
        (haiku_cumulative[mid_month - 1] + self_hosted_cumulative[mid_month - 1])
        / 2
        / 1e6
    )
    ax2.annotate(
        f"12-mo savings\nvs Haiku:\n${total_savings_haiku / 1e6:.2f}M",
        xy=(mid_month, mid_y),
        fontsize=10,
        fontweight="bold",
        ha="center",
        va="center",
        bbox=dict(
            boxstyle="round,pad=0.4", facecolor="white", edgecolor="gray", alpha=0.9
        ),
    )

    ax2.set_xlabel("Month", fontsize=11)
    ax2.set_ylabel("Cumulative Cost ($M)", fontsize=11)
    ax2.set_title("Cumulative Spend Over 12 Months", fontsize=12, fontweight="bold")
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:.1f}M"))
    ax2.set_xticks(months)
    ax2.legend(fontsize=9, loc="upper left")
    ax2.grid(True, alpha=0.25)

    plt.tight_layout()
    plt.show()

    # ── Summary ──
    final_volume = int(volumes[-1])
    print(
        f"\n  Growth: {starting_volume:,} \u2192 {final_volume:,} appeals/month over 12 months"
    )
    print(f"  ({monthly_growth_pct:.0f}% monthly growth)\n")
    print(f"  12-Month Cumulative Spend:")
    print(f"    Bedrock Haiku:   ${haiku_cumulative[-1]:>12,.0f}")
    print(f"    Bedrock Sonnet:  ${sonnet_cumulative[-1]:>12,.0f}")
    print(f"    Self-Hosted:     ${self_hosted_cumulative[-1]:>12,.0f}")
    print(f"\n  12-Month Savings vs Haiku:   ${total_savings_haiku:>12,.0f}")
    print(f"  12-Month Savings vs Sonnet:  ${total_savings_sonnet:>12,.0f}")

interactive(children=(IntSlider(value=300000, description='Starting Appeals/Mo', layout=Layout(width='600px'),…

---

## Key Takeaways

1. **Self-hosting is already cheaper at our initial target of 300K appeals/month** — the crossover happens well below that volume for the L40S configuration.

2. **API costs have zero economies of scale.** The cost per appeal is the same at 10K/month as it is at 10M/month. Self-hosted cost per appeal drops continuously.

3. **The savings compound rapidly with growth.** At 15% monthly growth starting from 300K, the 12-month savings vs. Haiku alone exceed **$500K+**, and the gap vs. Sonnet-quality is measured in millions.

4. **Infrastructure headroom is built-in.** Self-hosted clusters have spare capacity between step-ups, meaning we get burst absorption and latency headroom at no additional cost.

---

### Assumptions & Caveats

- Throughput estimates use a simplified roofline model (memory-bandwidth bound decode, compute-bound prefill) with a 65% hardware efficiency factor
- KV cache memory is estimated from model architecture (GQA heads, head dim, layers) at FP16 precision
- GPU pricing uses on-demand rates; Reserved Instances or Savings Plans reduce self-hosted costs by a further 30-60%
- Does not include one-time migration/engineering costs (amortized over 12 months, these are small relative to ongoing savings)
- Assumes open-source model quality meets product requirements (requires separate evaluation)
- Does not model spot instance strategies which could further reduce costs for batch workloads